# Konwersja dokumentacji Markdown → PDF

Notatnik zamienia plik `.md` na PDF o układzie dokumentu projektowego:

* **strona tytułowa** budowana z tabeli metadanych na początku pliku (tytuł projektu, autorzy, promotor, data, wersja),
* streszczenie, historia zmian i spis treści na osobnej stronie, a **każdy rozdział `## N.` od nowej strony**,
* stopka z tytułem dokumentu i numerem strony, zakładki PDF, klikalny spis treści,
* **diagram Gantta** rysowany lokalnie (matplotlib), pozostałe diagramy Mermaid przez serwis mermaid.ink (wymaga internetu; bez niego zostają jako kod).

Wymagane pakiety: `pip install -r reqirements.txt`

In [ ]:
from pathlib import Path

# --- Ustawienia ---
INPUT_MD = Path("dosc/Stages/etap-1-wymagania-i-zakres-new.md")
OUTPUT_PDF = Path("etap-1-wymagania-i-zakres-new.pdf")
AUTHORS = "Piotr Wysocki, Miłosz Zieliński"
TOC_LEVEL = 2          # poziom nagłówków w zakładkach PDF
RENDER_MERMAID = True  # False = diagramy inne niż Gantt zostają jako kod

## Funkcje konwersji

In [ ]:
import base64
import datetime as dt
import re
import shutil
import tempfile
import unicodedata
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pymupdf
import requests
from markdown_pdf import MarkdownPdf, Section
from mdit_py_plugins.anchors import anchors_plugin

# --- Wygląd -------------------------------------------------------------
ACCENT = "#1F3864"      # granat – nagłówki
ACCENT_LIGHT = "#DCE3F0"  # tło nagłówków tabel
MARGINS = (56, 56, -56, -60)  # lewy, górny, prawy, dolny (pt)

# Uwaga: bez background-color w tabelach i blokach kodu – MuPDF przy podziale
# tabeli na dwie strony rysuje wtedy na drugiej stronie „duchy” tła nagłówka.
CSS = f"""
body {{ font-family: sans-serif; font-size: 9.5pt; line-height: 1.35; color: #222; }}
h1 {{ font-size: 20pt; color: {ACCENT}; margin: 0 0 10pt 0; }}
h2 {{ font-size: 15pt; color: {ACCENT}; border-bottom: 1.5px solid {ACCENT};
      padding-bottom: 3pt; margin: 0 0 10pt 0; }}
h3 {{ font-size: 11.5pt; color: {ACCENT}; margin: 14pt 0 5pt 0; }}
p {{ margin: 0 0 6pt 0; text-align: justify; }}
table {{ border-collapse: collapse; width: 100%; margin: 4pt 0 10pt 0; font-size: 8.5pt; }}
th {{ color: {ACCENT}; font-weight: bold; text-align: left; padding: 3pt 4pt;
      border: 1px solid #B8C2D6; border-bottom: 2px solid {ACCENT}; }}
td {{ border: 1px solid #B8C2D6; padding: 3pt 4pt; vertical-align: top; }}
code {{ font-family: monospace; font-size: 8pt; color: #333; }}
pre {{ border: 1px solid #D5DBE6; padding: 5pt; font-size: 7.5pt; }}
blockquote {{ color: #444; border-left: 3px solid {ACCENT_LIGHT}; padding-left: 8pt; }}
ul, ol {{ margin: 2pt 0 6pt 0; }}
li {{ margin-bottom: 1pt; }}
a {{ color: {ACCENT}; text-decoration: none; }}
img {{ display: block; margin: 6pt auto; }}
"""

TITLE_CSS = f"""
body {{ font-family: sans-serif; color: #222; text-align: center; }}
.uni {{ font-size: 13pt; font-weight: bold; color: {ACCENT}; margin-top: 30pt; }}
.fac {{ font-size: 10.5pt; color: #444; margin-top: 3pt; }}
.kind {{ font-size: 10pt; color: #666; margin-top: 120pt; text-transform: uppercase; letter-spacing: 1pt; }}
.title {{ font-size: 24pt; font-weight: bold; color: {ACCENT}; margin-top: 10pt; line-height: 1.2; }}
.doc {{ font-size: 14pt; color: #333; margin-top: 14pt; }}
.line {{ border-top: 2px solid {ACCENT}; margin: 26pt 120pt 26pt 120pt; }}
.k {{ font-size: 8pt; color: #888; text-transform: uppercase; letter-spacing: 0.5pt; margin-top: 10pt; }}
.v {{ font-size: 11pt; color: #222; margin-top: 2pt; }}
"""

MAX_W, MAX_H = 480, 430  # maks. rozmiar diagramu na stronie (pt)


# --- Linki wewnętrzne -----------------------------------------------------
def slug(text: str) -> str:
    """Identyfikator nagłówka bez polskich znaków (np. 'Słownik pojęć' -> 'slownik-pojec')."""
    text = text.strip().lower().replace("ł", "l")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    text = re.sub(r"[^\w\- ]", "", text)
    return text.replace(" ", "-")


def fix_internal_links(text: str) -> str:
    return re.sub(r"\]\(#([^)]+)\)", lambda m: f"](#{slug(m.group(1))})", text)


# --- Diagramy ---------------------------------------------------------------
def _img_tag(path: Path, max_w=MAX_W, max_h=MAX_H) -> str:
    pix = pymupdf.Pixmap(str(path))
    scale = min(max_w / pix.width, max_h / pix.height)
    return f'\n<img src="{path.name}" width="{int(pix.width * scale)}" height="{int(pix.height * scale)}">\n'


def gantt_to_png(code: str, out: Path) -> None:
    """Rysuje diagram Gantta (składnia Mermaid) w matplotlib."""
    sections, title = [], ""
    for raw in code.splitlines():
        line = raw.strip()
        if line.startswith("title "):
            title = line[6:].strip()
        elif line.startswith("section "):
            sections.append((line[8:].strip(), []))
        elif ":" in line and sections and not line.startswith(("dateFormat", "axisFormat")):
            name, spec = line.split(":", 1)
            parts = [p.strip() for p in spec.split(",")]
            milestone = parts[0] == "milestone"
            if milestone:
                parts = parts[1:]
            start = dt.date.fromisoformat(parts[-2])
            end = start if milestone else dt.date.fromisoformat(parts[-1])
            sections[-1][1].append((name.strip(), start, end, milestone))

    rows = [(sec, *task) for sec, tasks in sections for task in tasks]
    colors = plt.cm.tab10.colors
    fig_h = 0.32 * len(rows) + 1.2
    fig, ax = plt.subplots(figsize=(10, fig_h), dpi=200)
    sec_names = [s for s, _ in sections]
    for i, (sec, name, start, end, milestone) in enumerate(rows):
        y = len(rows) - 1 - i
        c = colors[sec_names.index(sec) % 10]
        if milestone:
            ax.scatter(mdates.date2num(start), y, marker="D", s=55, color=ACCENT, zorder=3)
        else:
            ax.barh(y, (end - start).days + 1, left=mdates.date2num(start), height=0.6,
                    color=c, alpha=0.85, edgecolor="white")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[1] for r in reversed(rows)], fontsize=8)
    for lbl, r in zip(ax.get_yticklabels(), reversed(rows)):
        if r[4]:
            lbl.set_fontweight("bold"); lbl.set_color(ACCENT)
    ax.xaxis_date()
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=0))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="x", which="major", color="#999", linewidth=0.8)
    ax.grid(axis="x", which="minor", color="#ddd", linewidth=0.5)
    ax.set_axisbelow(True)
    # granice sekcji
    y = len(rows)
    for sec, tasks in sections:
        y -= len(tasks)
        ax.axhline(y - 0.5, color="#ccc", linewidth=0.6)
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors[i % 10], alpha=0.85) for i in range(len(sections))]
    ax.legend(handles, sec_names, fontsize=7, loc="upper right", frameon=False)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_ylim(-0.7, len(rows) - 0.3)
    if title:
        ax.set_title(title, fontsize=10, color=ACCENT, loc="left")
    fig.tight_layout()
    fig.savefig(out, facecolor="white")
    plt.close(fig)


def mermaid_ink_png(code: str, out: Path) -> None:
    encoded = base64.urlsafe_b64encode(code.encode("utf-8")).decode("ascii")
    resp = requests.get(f"https://mermaid.ink/img/{encoded}", params={"type": "png"}, timeout=30)
    resp.raise_for_status()
    pymupdf.Pixmap(resp.content)  # sprawdza, czy to poprawny obraz
    out.write_bytes(resp.content)


def render_diagrams(text: str, img_dir: Path, use_mermaid_ink: bool = True) -> str:
    """Gantt -> matplotlib (offline); pozostałe diagramy Mermaid -> mermaid.ink."""
    pattern = re.compile(r"^```mermaid[ \t]*\n(.*?)^```[ \t]*$", re.S | re.M)
    counter = {"n": 0}

    def replace(match):
        code = match.group(1)
        counter["n"] += 1
        out = img_dir / f"diagram_{counter['n']}.png"
        try:
            if code.lstrip().startswith("gantt"):
                gantt_to_png(code, out)
                print(f"  ✓ Diagram {counter['n']} (Gantt) narysowany")
                return _img_tag(out, max_w=MAX_W, max_h=600)
            if not use_mermaid_ink:
                return match.group(0)
            mermaid_ink_png(code, out)
            print(f"  ✓ Diagram {counter['n']} wyrenderowany")
            return _img_tag(out)
        except Exception as exc:
            print(f"  ! Diagram {counter['n']} nie został wyrenderowany ({type(exc).__name__}) – zostaje jako kod.")
            return match.group(0)

    return pattern.sub(replace, text)


# --- Struktura dokumentu ------------------------------------------------------
def _strip_hr(text: str) -> str:
    """Usuwa linie poziome '---' (podział na strony robią sekcje)."""
    return re.sub(r"^---[ \t]*$\n?", "", text, flags=re.M)


def split_document(text: str):
    """Zwraca: (blok tytułowy, wstęp: streszczenie/historia/spis, lista rozdziałów)."""
    parts = re.split(r"(?m)^(?=## )", text)
    head, rest = parts[0], parts[1:]
    front = [p for p in rest if not re.match(r"## \d", p)]
    chapters = [p for p in rest if re.match(r"## \d", p)]
    return head, "".join(front), chapters


def title_page_html(head: str, faculty: str, university: str) -> str:
    doc_title = re.search(r"^# (.+)$", head, re.M).group(1).strip()
    rows = re.findall(r"^\|\s*\*\*(.+?)\*\*\s*\|\s*(.+?)\s*\|\s*$", head, re.M)
    meta = dict(rows)
    project = meta.pop("Tytuł projektu", "")
    table = "".join(f'<div class="k">{k}</div><div class="v">{v}</div>' for k, v in meta.items())
    return f"""
<div class="uni">{university}</div>
<div class="fac">{faculty}</div>
<div class="kind">Praca dyplomowa inżynierska</div>
<div class="title">{project}</div>
<div class="doc">{doc_title}</div>
<div class="line"></div>
{table}
""", doc_title, project


def add_footer(pdf_path: Path, out_path: Path, left_text: str) -> None:
    doc = pymupdf.open(pdf_path)
    n = doc.page_count
    for i, page in enumerate(doc):
        if i == 0:
            continue  # bez stopki na stronie tytułowej
        r = page.rect
        page.draw_line((56, r.height - 42), (r.width - 56, r.height - 42), color=(0.6, 0.65, 0.75), width=0.6)
        style = "font-family: sans-serif; font-size: 7.5pt; color: #666;"
        page.insert_htmlbox(pymupdf.Rect(56, r.height - 38, r.width - 130, r.height - 20),
                            f'<div style="{style}">{left_text}</div>')
        page.insert_htmlbox(pymupdf.Rect(r.width - 130, r.height - 38, r.width - 56, r.height - 20),
                            f'<div style="{style} text-align: right;">Strona {i + 1} z {n}</div>')
    doc.save(out_path, garbage=3, deflate=True)
    doc.close()


def convert(input_md: Path, output_pdf: Path, toc_level=2, render_mermaid=True,
            authors="", university="Politechnika Warszawska",
            faculty="Wydział Matematyki i Nauk Informacyjnych"):
    text = fix_internal_links(input_md.read_text(encoding="utf-8"))
    img_dir = Path(tempfile.mkdtemp(prefix="md2pdf_"))
    try:
        text = render_diagrams(text, img_dir, use_mermaid_ink=render_mermaid)
        head, front, chapters = split_document(text)
        tp_html, doc_title, project = title_page_html(head, faculty, university)

        pdf = MarkdownPdf(toc_level=toc_level)
        pdf.m_d.use(anchors_plugin, min_level=1, max_level=6, slug_func=slug)
        pdf.meta.update(title=f"{project} – {doc_title}", author=authors)

        root = str(img_dir)
        pdf.add_section(Section(tp_html, toc=False, root=root, borders=MARGINS), user_css=TITLE_CSS)
        if front.strip():
            pdf.add_section(Section(_strip_hr(front), root=root, borders=MARGINS), user_css=CSS)
        for ch in chapters:
            pdf.add_section(Section(_strip_hr(ch), root=root, borders=MARGINS), user_css=CSS)

        # zakładki PDF muszą zaczynać się od poziomu 1 (tu najwyższy jest nagłówek ##)
        if pdf.toc:
            top = min(item[0] for item in pdf.toc)
            pdf.toc = [(lvl - top + 1, *rest) for lvl, *rest in pdf.toc]

        tmp_pdf = img_dir / "tmp.pdf"
        pdf.save(tmp_pdf)
        add_footer(tmp_pdf, output_pdf, doc_title)
    finally:
        shutil.rmtree(img_dir, ignore_errors=True)
    print(f"Zapisano: {Path(output_pdf).resolve()}")

## Konwersja

In [ ]:
convert(INPUT_MD, OUTPUT_PDF, toc_level=TOC_LEVEL, render_mermaid=RENDER_MERMAID, authors=AUTHORS)